# 04 Self-Attention 矩阵形式与形状变化

上一节我们用“一个位置去看所有位置”的方式理解了 Self-Attention。

这一节把它改写成矩阵形式。

你会看到这个公式：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

先别急着背。

这一节只做一件事：

```text
把这个公式每一步的形状变化讲清楚。
```

只要形状能跟上，公式就不会像一团雾。

## 1. 为什么要学矩阵形式

上一节我们是这样讲的：

```text
第 i 个位置用 q_i 去匹配所有 k。
得到第 i 行注意力权重。
再用这行权重汇总所有 v。
```

这种讲法适合理解概念。

但真实计算时，模型不会一个位置一个位置慢慢算。

它会把所有位置一起放进矩阵里，一次性完成计算。

所以矩阵形式不是新机制。

它只是把上一节的很多次小计算合并成一次大计算。

## 2. 先用一个固定例子

为了避免全是字母，我们先固定一个例子。

假设一句话有 4 个 token：

```text
N = 4
```

每个 token 原来用 8 维向量表示：

```text
D = 8
```

这一句话的输入形状就是：

```text
X：4 x 8
```

这里先不考虑 batch。

可以把 $X$ 理解成一张表：

```text
第 1 行：token 1 的向量
第 2 行：token 2 的向量
第 3 行：token 3 的向量
第 4 行：token 4 的向量
```

所以：

```text
行数 = token 数量
列数 = 每个 token 的特征维度
```

## 3. 从 X 生成 Q、K、V

Self-Attention 会用三组线性变换生成 Q、K、V。

假设：

```text
输入 X：4 x 8
W_Q：8 x 6
W_K：8 x 6
W_V：8 x 8
```

那么：

$$
Q=XW_Q
$$

形状变化是：

```text
4 x 8  @  8 x 6  ->  4 x 6
```

所以：

```text
Q：4 x 6
```

同理：

```text
K：4 x 6
V：4 x 8
```

注意这里的含义：

```text
Q 和 K 都是 4 x 6，因为它们要互相做点积匹配。
V 是 4 x 8，因为它决定最后汇总出来的内容维度。
```

## 4. 为什么要计算 QK^T

上一节说过，每个 Query 都要看所有 Key。

现在 Q 里面有 4 行：

```text
q_1
q_2
q_3
q_4
```

K 里面也有 4 行：

```text
k_1
k_2
k_3
k_4
```

我们想得到的是：

```text
q_1 和 k_1, k_2, k_3, k_4 的分数
q_2 和 k_1, k_2, k_3, k_4 的分数
q_3 和 k_1, k_2, k_3, k_4 的分数
q_4 和 k_1, k_2, k_3, k_4 的分数
```

这正好可以用矩阵乘法：

$$
QK^T
$$

其中 $K^T$ 表示把 K 转置。

## 5. QK^T 的形状怎么来

现在看形状。

Q 的形状是：

```text
Q：4 x 6
```

K 的形状是：

```text
K：4 x 6
```

如果直接用 Q 乘 K，形状对不上。

所以要把 K 转置：

```text
K^T：6 x 4
```

于是：

```text
QK^T：4 x 6  @  6 x 4  ->  4 x 4
```

这个 `4 x 4` 就是注意力分数表。

行表示谁在看。

列表示被看的是谁。

```text
第 1 行：token 1 看所有 token 的分数
第 2 行：token 2 看所有 token 的分数
第 3 行：token 3 看所有 token 的分数
第 4 行：token 4 看所有 token 的分数
```

## 6. 为什么 Q 和 K 的最后一维必须一致

这里再强调一个关键点。

Q 和 K 之所以都设成最后一维 6，是因为它们要做点积。

一个 Query 向量是 6 维：

```text
q_i：1 x 6
```

一个 Key 向量也是 6 维：

```text
k_j：1 x 6
```

两个 6 维向量才能对应位置相乘再求和，得到一个分数。

如果 Q 是 6 维，K 是 5 维，就没法直接点积。

所以：

```text
d_k 同时表示 Query 和 Key 用来匹配的维度。
```

## 7. 除以 sqrt(d_k) 不改变形状

QK^T 得到的是分数表：

```text
4 x 4
```

公式里会除以：

$$
\sqrt{d_k}
$$

在这个例子里：

```text
d_k = 6
```

所以是把整个分数表除以 $\sqrt{6}$。

这一步只是缩放分数，不改变形状。

```text
4 x 4  ->  4 x 4
```

它的目的可以先理解成：

```text
让分数不要太大，避免 Softmax 过于极端。
```

## 8. Softmax 也不改变形状

缩放后的分数表形状是：

```text
4 x 4
```

接下来对每一行做 Softmax。

为什么是一行一行做？

因为每一行表示一个 token 在看所有 token 的分数。

一行 Softmax 后，就得到这个 token 看所有 token 的权重分布。

所以：

```text
分数表：4 x 4
权重表：4 x 4
```

Softmax 改变的是数值含义，不改变形状。

```text
原来每行是原始分数。
现在每行是注意力权重，非负，并且每行加起来等于 1。
```

## 9. 为什么最后要乘 V

现在我们有注意力权重表：

```text
A：4 x 4
```

这里用 $A$ 表示 Softmax 后的权重表。

V 的形状是：

```text
V：4 x 8
```

最后计算：

$$
AV
$$

形状变化是：

```text
4 x 4  @  4 x 8  ->  4 x 8
```

这个结果就是 Self-Attention 的输出。

每一行都是一个 token 更新后的新表示。

```text
第 1 行：token 1 汇总上下文后的表示
第 2 行：token 2 汇总上下文后的表示
第 3 行：token 3 汇总上下文后的表示
第 4 行：token 4 汇总上下文后的表示
```

## 10. 为什么输出还是 4 x 8

输入是：

```text
X：4 x 8
```

输出也是：

```text
O：4 x 8
```

这很常见。

但这不表示什么都没变。

形状一样，只说明：

```text
仍然有 4 个位置。
每个位置仍然用 8 维向量表示。
```

真正变化的是每个位置向量的内容。

原来的 $\mathbf{x}_i$ 主要是 token 自己的表示。

新的 $\mathbf{o}_i$ 已经融合了其他 token 的信息。

所以可以记成：

```text
形状可以不变，但信息已经融合了上下文。
```

## 11. 把完整形状路线串起来

现在把这一节的例子完整串起来。

```text
输入 X：4 x 8

生成 Q：4 x 8 @ 8 x 6 -> 4 x 6
生成 K：4 x 8 @ 8 x 6 -> 4 x 6
生成 V：4 x 8 @ 8 x 8 -> 4 x 8

计算分数：QK^T = 4 x 6 @ 6 x 4 -> 4 x 4
缩放分数：4 x 4 -> 4 x 4
Softmax：4 x 4 -> 4 x 4
加权汇总：4 x 4 @ 4 x 8 -> 4 x 8

输出 O：4 x 8
```

这就是 Self-Attention 的形状变化主线。

如果你能把这条线说出来，矩阵公式就基本不吓人了。

## 12. 推广到一般符号

刚才用的是具体数字。

现在换成一般符号。

不考虑 batch 时，输入是：

```text
X：N x D
```

设：

```text
W_Q：D x d_k
W_K：D x d_k
W_V：D x d_v
```

那么：

```text
Q：N x d_k
K：N x d_k
V：N x d_v
```

分数表：

```text
QK^T：N x d_k @ d_k x N -> N x N
```

权重表：

```text
Softmax 后：N x N
```

输出：

```text
N x N @ N x d_v -> N x d_v
```

## 13. 加上 batch 后怎么办

真实训练时通常是一批样本一起算。

输入形状是：

```text
X：B x N x D
```

经过线性变换后：

```text
Q：B x N x d_k
K：B x N x d_k
V：B x N x d_v
```

每个样本内部都会计算自己的注意力表。

所以注意力分数形状是：

```text
B x N x N
```

输出形状是：

```text
B x N x d_v
```

这里要记住：

```text
batch 维 B 只是表示有多少个样本并行计算。
每个样本内部仍然是 N 个位置互相看。
```

## 14. 为什么注意力表不是 N x d_k

这是一个常见混淆点。

Q 和 K 的形状都是：

```text
N x d_k
```

但注意力分数表不是 $N\times d_k$。

因为分数表记录的是“位置和位置之间的关系”。

有 $N$ 个 Query，也有 $N$ 个 Key。

每个 Query 都要和每个 Key 算一个分数。

所以分数表是：

```text
N x N
```

$d_k$ 只是每次匹配时用到的向量维度。

它会在点积时被消掉。

```text
N x d_k @ d_k x N -> N x N
```

## 15. 为什么乘 V 后不是 N x N

另一个容易混的地方是最后一步。

注意力权重表是：

```text
N x N
```

V 是：

```text
N x d_v
```

相乘后：

```text
N x N @ N x d_v -> N x d_v
```

为什么不是 $N\times N$？

因为乘 V 的目的不是继续保存“谁看谁”的关系表。

它的目的是用每一行权重去汇总 Value 内容。

所以输出重新变成“每个位置一个向量”：

```text
N 个位置，每个位置 d_v 维。
```

## 16. 从一句话到上下文化表示：把含义重新串起来

现在把前面的矩阵计算翻译回一条完整的自然语言处理流程。

第一步，一句话先被切分成 $N$ 个 token。每个 token 经过 Embedding，再加入位置编码，得到自己的初始向量。把这些向量按 token 顺序排列，就得到输入矩阵：

```text
X：N x D
```

这里的每一行对应一个 token。第一层中的 $X$ 主要包含 token 本身和位置信息；在更深的层中，$X$ 则来自上一层的输出。

第二步，同一个 $X$ 经过三组不同的可学习变换：

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

可以把它们理解成同一个 token 的三种身份：

```text
Query：为了更新当前位置，我需要寻找什么信息？
Key：我有哪些特征可以用来和这种需求匹配？
Value：如果当前位置关注我，我能提供什么内容？
```

这里的“问题”“特征”和“内容”都只是帮助理解的自然语言说法。模型不会先写出一句文字问题，再把它编码成 Query；模型直接通过 $W_Q$、$W_K$、$W_V$ 生成向量。这三组参数开始时通常没有明确含义，而是在训练和反向传播中逐渐学出来的。

第三步，Q 和 K 负责算出谁应该关注谁，V 负责提供真正被汇总的内容：

```text
QK^T：算匹配分数
除以 sqrt(d_k)：缩放分数
Softmax：得到注意力权重 A
AV：按照权重汇总 Value 内容
```

最终仍然得到 $N$ 个向量，但每个向量已经融合了其他 token 的信息。直觉上可以把它看成一次新的表示，严格一些通常叫作 token 的上下文化表示，而不是最初的 token embedding。

## 17. 注意力表中的格子怎样和 Value 对应

注意力权重表中的每个格子 $a_{i,j}$ 都是一个标量：

```text
a_ij：第 i 个 token 对第 j 个 token 的关注程度
```

第 $j$ 个 Key 和第 $j$ 个 Value 来自同一个 token，所以第 $j$ 列的权重总是对应第 $j$ 个 Value：

```text
第 1 列 -> v_1
第 2 列 -> v_2
...
第 N 列 -> v_N
```

因此，不是注意力表的每个格子都有一条独立的 Value 向量。同一列的所有格子对应同一个 token 的 Value，只是不同 Query 给它的权重不同。

对于第 $i$ 行，输出是：

$$
\mathbf{o}_i=a_{i,1}\mathbf{v}_1+a_{i,2}\mathbf{v}_2+\cdots+a_{i,N}\mathbf{v}_N
$$

也就是说，一行有 $N$ 个权重，每个权重乘对应 token 的整条 Value 向量，最后把结果相加，得到当前 token 的新向量。

例如 $N=3$、$d_v=2$：

```text
A：3 x 3
V：3 x 2
O：3 x 2

(3 x 3) @ (3 x 2) -> (3 x 2)
```

$d_v$ 决定每条 Value 向量有多少维，由模型结构设定。它可以是 2、8 或 64。矩阵相乘真正要求相等的是中间的 $N$：注意力表有 $N$ 列，Value 矩阵有 $N$ 行，因为两边都按同样的 token 顺序排列。

所以最后乘 V 的根本原因是：

```text
注意力权重只表示“谁重要、重要多少”。
乘 V 才会按照这种重要性真正取出并融合内容。
```

## 18. 为什么要除以 sqrt(d_k)

$d_k$ 是一个 Query 或 Key 向量的维度。两个向量做点积时，需要把 $d_k$ 项乘积加起来：

$$
\mathbf{q}\cdot\mathbf{k}=q_1k_1+q_2k_2+\cdots+q_{d_k}k_{d_k}
$$

维度越大，相加的项越多，点积分数的波动通常也越大。可以把每一项暂时想成随机的 $+1$ 或 $-1$：64 项会互相抵消，结果的典型大小不是 64，而大约是 $\sqrt{64}=8$。这和随机走 64 步后，离起点的典型距离大约是 8 步类似。

因此，点积分数的典型尺度会随着 $\sqrt{d_k}$ 增长。把分数除以 $\sqrt{d_k}$，可以把不同维度下的分数拉回相对稳定的范围：

$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

为什么要让分数稳定？因为下一步是 Softmax。如果输入 Softmax 的分数过大，它很容易过早地产生接近 1 和 0 的极端权重，使训练中的梯度变小、学习不稳定。

可以把除以 $\sqrt{d_k}$ 类比成一种尺度调整，但要和严格的归一化、标准化区分：

```text
除以 sqrt(d_k)：只用固定常数缩放分数，不改变形状。
Softmax：把每一行变成非负、总和为 1 的注意力权重。
标准化：通常还涉及减均值、除以标准差，这里没有做。
```

所以最准确的记法是：$\sqrt{d_k}$ 负责给点积分数“降温”，Softmax 才负责把它们变成归一化的权重。

## 19. 本节小结

这一节先记住这条形状路线：

```text
X：N x D

Q：N x d_k
K：N x d_k
V：N x d_v

QK^T：N x N
除以 sqrt(d_k)：N x N
Softmax 后的 A：N x N
乘 V：N x d_v
```

再记住含义：

```text
Q 表示当前位置需要寻找什么。
K 表示每个位置有哪些可供匹配的特征。
V 表示每个位置能够提供的内容。
QK^T 负责算位置之间的分数表。
除以 sqrt(d_k) 负责控制分数尺度。
Softmax 把每一行分数变成权重。
AV 用每一行权重汇总所有 Value。
输出仍然是每个位置一个向量，但已经融合了上下文。
```

## 20. 自测问题

1. 为什么矩阵形式不是新的 Attention 机制？
2. 如果输入 $X$ 是 `4 x 8`，$W_Q$ 是 `8 x 6`，那么 Q 的形状是什么？
3. 为什么 K 要转置成 $K^T$ 才能和 Q 相乘？
4. `QK^T：4 x 6 @ 6 x 4 -> 4 x 4` 中，`4 x 4` 表示什么？
5. 注意力分数表里，行和列分别表示什么？
6. 为什么除以 $\sqrt{d_k}$ 不改变形状？
7. Softmax 为什么通常按行做？
8. 为什么 Softmax 后仍然是 `N x N`？
9. 为什么最后要乘 V？
10. 为什么 `N x N @ N x d_v` 会得到 `N x d_v`？
11. 如果加上 batch，输入 `B x N x D` 最后通常输出什么形状？
12. 为什么形状一样不代表信息没有变化？
13. 为什么不能说注意力表里的每个格子都有一个独立的 Value？
14. 注意力表第 $j$ 列为什么会与 $\mathbf{v}_j$ 对应？
15. $d_v$ 决定 Value 矩阵的哪一维？为什么它不必等于 $N$？
16. 为什么点积分数的典型尺度会随着 $\sqrt{d_k}$ 增长？
17. 除以 $\sqrt{d_k}$ 与 Softmax 分别负责什么？